In [2]:
# ── ACCOUNT 2: Full setup + Stage 2 ──────────────────────────────────────────

!pip install -q transformers datasets scikit-learn huggingface_hub

from datasets import load_dataset
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from huggingface_hub import login
import torch, torch.nn as nn, numpy as np, gc, shutil

# ── Data ──────────────────────────────────────────────────────────────────────
dataset  = load_dataset("freococo/burmese-contextual-pragmatics")
train_ds = dataset["train"]

politeness_map = {
    "neutral":"neutral","polite":"polite","informal":"informal",
    "professional":"professional","blunt":"blunt","rude":"rude",
    "very_polite":"polite","friendly":"informal","religious":"polite",
    "impolite":"rude","stern":"blunt","sarcastic":"rude","polite_but_firm":"polite"
}

def apply_labels(example):
    example["politeness_coarse"] = politeness_map.get(example["politeness"],"neutral")
    return example

train_ds = train_ds.map(apply_labels)
le = LabelEncoder().fit(train_ds["politeness_coarse"])

def encode_labels(example):
    example["label"] = int(le.transform([example["politeness_coarse"]])[0])
    return example

train_ds = train_ds.map(encode_labels)

split1     = train_ds.train_test_split(test_size=0.30, seed=42)
split2     = split1["test"].train_test_split(test_size=0.50, seed=42)
train_data = split1["train"]
val_data   = split2["train"]
test_data  = split2["test"]

counts  = Counter(train_data["label"])
total   = len(train_data)
n       = len(le.classes_)
weights = torch.tensor([total/(n*counts[i]) for i in range(n)], dtype=torch.float)

# ── Tokenizer ─────────────────────────────────────────────────────────────────
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(data, fn):
    keep = ["input_ids","attention_mask","label"]
    def wrapped(example):
        enc = fn(example)
        enc["label"] = example["label"]
        return enc
    out = data.map(wrapped, remove_columns=[c for c in data.column_names if c not in keep])
    out.set_format("torch")
    return out

def tok_stage2(example):
    text = (example["utterance"]
            + " </s> " + str(example["context"]).strip()
            + " </s> " + str(example["instruction"]).strip())
    return tokenizer(text, max_length=MAX_LENGTH, truncation=True, padding="max_length")

# ── Trainer ───────────────────────────────────────────────────────────────────
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        loss    = nn.CrossEntropyLoss(weight=self.class_weights.to(outputs.logits.device))(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

from sklearn.metrics import accuracy_score, f1_score
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy":    round(accuracy_score(labels, preds), 4),
        "macro_f1":    round(f1_score(labels, preds, average="macro"), 4),
        "weighted_f1": round(f1_score(labels, preds, average="weighted"), 4),
    }

# ── Login ─────────────────────────────────────────────────────────────────────
login()

# ── Tokenize ──────────────────────────────────────────────────────────────────
print("Tokenizing...")
tok_train = tokenize(train_data, tok_stage2)
tok_val   = tokenize(val_data,   tok_stage2)
tok_test  = tokenize(test_data,  tok_stage2)
print("Done.")

# ── Train ─────────────────────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=n)

args = TrainingArguments(
    output_dir="annasus10/xlmr-burmese-pragmatics-stage2-v2",
    num_train_epochs=8, per_device_train_batch_size=8,
    per_device_eval_batch_size=16, learning_rate=2e-5,
    warmup_steps=150, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch",
    save_total_limit=1, load_best_model_at_end=True,
    metric_for_best_model="macro_f1", logging_steps=50,
    report_to="none", seed=42, push_to_hub=True,
    hub_model_id="annasus10/xlmr-burmese-pragmatics-stage2-v2",
    hub_strategy="every_save"
)

trainer = WeightedTrainer(
    class_weights=weights, model=model, args=args,
    train_dataset=tok_train, eval_dataset=tok_val,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.push_to_hub()
print("Stage 2 pushed.")

# ── Evaluate ──────────────────────────────────────────────────────────────────
results = trainer.evaluate(tok_test)
print("\nStage 2 Test Results:")
for k, v in results.items():
    print(f"  {k}: {v}")

preds_out = trainer.predict(tok_test)
preds = np.argmax(preds_out.predictions, axis=1)
print(classification_report(preds_out.label_ids, preds, target_names=le.classes_))

Tokenizing...


Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Done.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

New Data Upload               : |          |  0.00B /  0.00B            

  ...age2-v2/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...age2-v2/model.safetensors:   9%|8         | 95.3MB / 1.11GB            

Stage 2 pushed.



Stage 2 Test Results:
  eval_loss: 0.8035444617271423
  eval_accuracy: 0.7939
  eval_macro_f1: 0.7583
  eval_weighted_f1: 0.7943
  eval_runtime: 2.0509
  eval_samples_per_second: 160.907
  eval_steps_per_second: 10.24
  epoch: 8.0
              precision    recall  f1-score   support

       blunt       0.73      0.79      0.76        14
    informal       0.75      0.76      0.76        59
     neutral       0.88      0.79      0.83       174
      polite       0.64      0.88      0.74        56
professional       0.88      0.93      0.90        15
        rude       0.83      0.42      0.56        12

    accuracy                           0.79       330
   macro avg       0.79      0.76      0.76       330
weighted avg       0.81      0.79      0.79       330

